In [161]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import networkx as nx
import sqlite3

con = sqlite3.connect("data/broadband.db")

RESET = False

In [162]:
if RESET:
    con.execute("DROP TABLE IF EXISTS fcc_tracts")
    con.execute("""
        CREATE TABLE fcc_tracts (
            geoid TEXT PRIMARY KEY CHECK (length(geoid) = 11),
            county TEXT NOT NULL,
            total_locations INTEGER NOT NULL CHECK (total_locations >= 0),
            no_terra_service INTEGER NOT NULL CHECK (no_terra_service BETWEEN 0 AND total_locations),
            pct_without_coverage REAL
        ) STRICT
    """)

    con.execute("DROP TABLE IF EXISTS tract_clusters")
    con.execute("""
        CREATE TABLE tract_clusters (
            geoid TEXT NOT NULL CHECK (length(geoid) = 11), 
            threshold INTEGER NOT NULL CHECK (threshold BETWEEN 0 AND 100),
            cluster INTEGER NOT NULL CHECK (cluster >= 1),
            cluster_rank INTEGER NOT NULL CHECK (cluster_rank >= 1),
            PRIMARY KEY (geoid, threshold)
        ) STRICT
    """)

    con.commit()

In [163]:
cluster_nosub_vs_noterra = pd.read_sql_query("""
    SELECT 
        SUM(s.total) AS households,
        ROUND(100.0 * (SUM(s.access_no_sub) + SUM(s.no_access)) / SUM(s.total), 2) AS pct_households_no_sub,
        c.cluster_rank AS rank,
        COUNT(*)                AS tracts,
        SUM(f.total_locations)  AS locations,
        SUM(f.no_terra_service) AS no_terra_service,
        ROUND(100.0 * SUM(f.no_terra_service) / SUM(f.total_locations), 2) AS pct,
        group_concat(DISTINCT f.county) AS counties
    FROM tract_clusters AS c, acs_subs AS s
    JOIN fcc_tracts AS f ON f.geoid = c.geoid AND f.geoid = s.geoid
    WHERE c.threshold = 15
    GROUP BY c.cluster_rank
    ORDER BY c.cluster_rank
""", con)

cluster_nosub_vs_noterra.to_excel(f"results/Clusters of Flagged Tracts.xlsx", index=False)

In [164]:
flagged_tract_demographics = pd.read_sql_query("""
    SELECT CASE WHEN c.geoid IS NULL THEN 'not flagged' ELSE 'flagged' END AS grp,
        COUNT(*) AS tracts,
        SUM(i.total) AS households,
        SUM(a.total) AS people,
        SUM(a.age_u18) AS children,
        SUM(l.lf) AS labor_force,
        ROUND(100.0 * SUM(i.inc_lt_10k + i.inc_10_20k + i.inc_20_35k) / SUM(i.total), 2) AS pct_households_earning_lt35k,
        ROUND(100.0 * SUM(age_65_plus) / SUM(a.total), 2) AS pct_aged_65_plus,
        ROUND(100.0 * SUM(age_u18_computer_no_sub + age_u18_no_computer) / SUM(a.age_u18), 2) AS pct_children_no_broadband,
        ROUND(100.0 * SUM(lf_unemployed) / SUM(l.lf), 2) AS pct_labor_froce_unemployed
    FROM fcc_tracts AS f
    LEFT JOIN tract_clusters AS c ON c.geoid = f.geoid AND c.threshold = 15
    LEFT JOIN acs_income AS i ON i.geoid = f.geoid
    LEFT JOIN acs_age AS a ON a.geoid = f.geoid
    LEFT JOIN acs_labor AS l ON l.geoid = f.geoid
    GROUP BY grp
""", con)

flagged_tract_demographics.to_excel("results/Flagged vs. Not Flagged Tracts Demographics Analysis.xlsx", index=False)

In [165]:
con.close()